In [0]:
from pyspark.sql.functions import col, round, when

bronze_df = spark.table("workspace.default.sales_demo")

silver_df = (
    bronze_df
    .filter(col("sale_id").isNotNull())
    .filter(col("quantity") > 0)
    .filter(col("unit_price") >= 0)
    .dropDuplicates(["sale_id"])
    .withColumn(
        "sales_region",
        when(col("sale_id") % 2 == 0, "East")
        .otherwise("West")
    )
    .withColumn(
        "line_total",
        round(col("quantity") * col("unit_price"), 2)
    )
)

display(silver_df)

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_sales")

print("Created workspace.default.silver_sales")